# Import utils

In [1]:
from utils.classification_utils import ClassifierPreprocessor
from utils.dataset_utils import SegmentDatasetAllTimestamps

In [2]:
machine_part ="machine_and_movement"

# Create my own classifier class

In [3]:
import json

with open("../../../../data/ml/unique_experiment_ids.json", "r") as f:
    experiment_groups = json.load(f)

# Read and Preprocess for training

In [4]:
import matplotlib.pyplot as plt
import joblib
import torch
from utils.model import LSTMClassifier

def get_all_predictions(Label, machine_part="machine_and_movement", exp_id= 2):
    classifier_preprocessor = ClassifierPreprocessor(sensors_path=f"../../../../data/ml/features_{machine_part}_complete.csv", annotation_json="../../../../data/ml/machine-and-movement_complete.json")
    sensors_df, _ = classifier_preprocessor.read_data()
    sensors_df = classifier_preprocessor.feature_selection()
    sensors_df = classifier_preprocessor.assign_one_label(target_label=Label)
    sensors_df = classifier_preprocessor.normalize_and_encode_labels()

    feature_cols = classifier_preprocessor.get_feature_cols()
    input_size = len(feature_cols)
    hidden_size = 64
    num_layers = 2
    num_classes = 2
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LSTMClassifier(input_size, hidden_size, num_layers, num_classes, bidirectonal=True).to(device)
    if machine_part == "movement":
        machine_part ="movements"
    model_path = f"../../../../models/classifier/{machine_part}/{Label}"

    state_dict = joblib.load(f"{model_path}/Activity_Detector.joblib")
    model.load_state_dict(state_dict)
    model.eval()


    # --- Select a random experiment ---
    exp_data = sensors_df[sensors_df["Experiment_ID"] == exp_id]

    # --- Features ---
    X = torch.tensor(exp_data[feature_cols].values, dtype=torch.float32).to(device)
    X = X.unsqueeze(0)  # [1, seq_len, num_features]

    # --- Model predictions ---
    with torch.no_grad():
        outputs = model(X)
        y_pred = torch.argmax(outputs, dim=-1).squeeze(0).cpu().numpy()

    # Example boolean masks
    mask_decalmping_pred = (y_pred == 0)
    mask_declamping_true = (exp_data["Label_encoded"].values == 0)
    return exp_data, mask_decalmping_pred, mask_declamping_true

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display

Labels = ["Clamping", "Bending", "Mandrel Extraction", "De-Clamping"]

# Assign a distinct color for each label
base_colors = list(mcolors.TABLEAU_COLORS.values())[:len(Labels)]

# List of experiment IDs
test_exps = [2, 3, 22, 23, 40, 54, 83, 85, 110, 112, 119, 120, 121, 122, 123, 178,
             179, 182, 183, 211, 212, 213, 255, 258, 261, 271, 272, 273, 302, 303,
             304, 317, 318]

# --- Widget for selecting experiment ID ---
exp_widget = widgets.Dropdown(
    options=test_exps,
    value=test_exps[0],
    description='Experiment ID:',
    disabled=False,
)

def plot_experiment(exp_id):
    # --- Load all data upfront ---
    all_data = {}
    for Label in Labels:
        exp_data, mask_pred, mask_true = get_all_predictions(Label=Label, exp_id=str(exp_id))
        all_data[Label] = {"exp_data": exp_data, "mask_pred": mask_pred, "mask_true": mask_true}

    # --- Create figure ---
    fig, axs = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

    # --- Top subplot: sensor signals ---
    sensor_plotted = set()
    for Label in Labels:
        exp_data = all_data[Label]["exp_data"]
        sensor_cols = [col for col in exp_data.columns if col not in ["Label", "Label_encoded", "Experiment_ID"]]

        for col in sensor_cols:
            if col not in sensor_plotted:
                axs[0].plot(exp_data[col].values, label=col)
                sensor_plotted.add(col)
            else:
                axs[0].plot(exp_data[col].values)

    axs[0].set_ylabel("Sensor Value")
    axs[0].set_title(f"Sensor Signals - Experiment {exp_id}")
    axs[0].grid(True, linestyle='--', alpha=0.5)

    # Move legend outside the plot
    axs[0].legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize='small')

    # --- Bottom subplot: segments ---
    categories = []
    starts = []
    ends = []
    colors = []

    for i, Label in enumerate(Labels):
        mask_true = all_data[Label]["mask_true"]
        mask_pred = all_data[Label]["mask_pred"]

        base_color = base_colors[i]
        bright_color = mcolors.to_rgba(base_color, alpha=0.7)   # Bright for True
        dark_color = mcolors.to_rgba(base_color, alpha=0.3)     # Dark for Predicted

        # True segments
        start = None
        for j, val in enumerate(mask_true):
            if val == 1 and start is None:
                start = j
            elif val == 0 and start is not None:
                categories.append(f"{Label} True")
                starts.append(start)
                ends.append(j-1)
                colors.append(bright_color)
                start = None
        if start is not None:
            categories.append(f"{Label} True")
            starts.append(start)
            ends.append(len(mask_true)-1)
            colors.append(bright_color)

        # Predicted segments
        start = None
        for j, val in enumerate(mask_pred):
            if val == 1 and start is None:
                start = j
            elif val == 0 and start is not None:
                categories.append(f"{Label} Pred")
                starts.append(start)
                ends.append(j-1)
                colors.append(dark_color)
                start = None
        if start is not None:
            categories.append(f"{Label} Pred")
            starts.append(start)
            ends.append(len(mask_pred)-1)
            colors.append(dark_color)

    if categories:
        axs[1].barh(categories, [e-s for s, e in zip(starts, ends)], left=starts,
                    color=colors, height=0.6)
    else:
        axs[1].text(0.5, 0.5, 'No Segments Found', transform=axs[1].transAxes,
                    ha='center', va='center')

    plt.tight_layout()
    plt.show()

# --- Display widget ---
widgets.interact(plot_experiment, exp_id=exp_widget)


interactive(children=(Dropdown(description='Experiment ID:', options=(2, 3, 22, 23, 40, 54, 83, 85, 110, 112, …

<function __main__.plot_experiment(exp_id)>